# 코드 설명

- **코드 파일 사용 목차** : PE 헤더 데이터를 표준화하기
- **해당 코드에서 사용하는 CSV 파일** : `file_pe_headers.csv`
    - 규모 : 19,611행 × 79열
    - 내용 : Windows 실행 파일(.exe, .dll)의 PE 헤더 필드를 추출한 데이터
    - 목표 변수 : `Malware` (1이면 악성, 0이면 정상)
    - 식별자 열 : `Name` (파일명)
    - 설명 변수 : 나머지 77열. 전부 수치형
    - 결측 : 없음
    - 참고 : 악성 14,599개 / 정상 5,012개로 범주 비율이 약 3:1로 치우침
- **사용 라이브러리**
    - `pandas` (as pd) — CSV 적재 및 표 형식 데이터 처리
    - `sklearn` — 전처리
- **사용되는 함수**
    - `pd.read_csv("파일경로", sep=",")`
        - 함수의 목적 : CSV 파일을 `DataFrame` 객체로 적재
        - 함수의 파라미터
            - `filepath_or_buffer` : 파일 경로. 첫 번째 위치 인자
            - `sep` : 열 구분자. 기본값이 `,`이므로 본 코드에서는 생략 가능
    - `DataFrame.drop(labels, axis)`
        - 함수의 목적 : 지정한 행 또는 열을 제거한 새 `DataFrame` 반환
        - 함수의 파라미터
            - `labels` : 제거 대상 이름. 여러 개는 리스트로 전달
            - `axis` : `0`이면 행, `1`이면 열. 기본값 `0`
    - `DataFrame.to_numpy()`
        - 함수의 목적 : `DataFrame`을 NumPy 배열(`ndarray`)로 변환
        - 주의 : 괄호 누락 시 변환이 실행되지 않고 함수 객체 자체가 반환
    - `StandardScaler()`
        - 함수의 목적 : 각 열을 평균 0, 표준편차 1로 변환하는 객체 생성
        - 주요 메서드
            - `fit(X)` : 열별 평균과 표준편차를 계산해 객체 내부에 저장
            - `transform(X)` : 저장된 값을 이용해 변환 수행
            - `fit_transform(X)` : 위 두 단계를 연속 수행
- **실행 순서** : 적재 → 설명 변수 X 구성 → 표준화

In [1]:
# ── Google Colab 준비 셀 ─────────────────────────────────────────
# 로컬(VS Code, JupyterLab)에서 실행하면 아무 동작 없이 넘어갑니다.
# Colab에서 실행하면 저장소를 내려받아 실습 폴더로 이동합니다.
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    저장소 = "https://github.com/isl-hedgehog/python-cyber-seminar.git"
    이름 = 저장소.rsplit("/", 1)[-1].removesuffix(".git")
    실습폴더 = "01-data-preprocessing"

    if not os.path.isdir(f"/content/{이름}"):
        subprocess.run(["git", "clone", "-q", 저장소], cwd="/content", check=True)
    os.chdir(f"/content/{이름}/{실습폴더}")

    # Colab에는 pandas, numpy, scikit-learn이 이미 설치되어 있습니다.
    # 추가 패키지가 필요한 회차에서는 아래 줄의 주석을 해제하세요.
    # subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)

    print("Colab 준비 완료 :", os.getcwd())
else:
    print("로컬 환경 — 준비 셀을 건너뜁니다.")

로컬 환경 — 준비 셀을 건너뜁니다.


In [2]:
import pandas as pd

data = pd.read_csv("file_pe_headers.csv", sep=",")
X = data.drop(["Name", "Malware"], axis=1).to_numpy()

### 1. PE 헤더 데이터 적재 및 X 구성

**데이터의 성격**

`file_pe_headers.csv`는 Windows 실행 파일의 **PE(Portable Executable) 헤더** 값을 추출해 표로 정리한 데이터. 파일을 실행하지 않고 구조 정보만으로 악성 여부를 판별하는 **정적 분석** 방식에 사용.

표본 19,611개, 설명 변수 77개이며 목표 변수 `Malware`가 0 또는 1이므로 **이진 분류** 과제에 해당.

**두 열을 제외하는 이유**

`drop` 대상은 2개 열이지만 제외 근거는 서로 다름.

| 열 | 제외 사유 |
|---|---|
| `Malware` | 정답 열. X에 포함되면 모델이 답을 그대로 보게 됨 |
| `Name` | 파일명. 각 행을 구분하는 이름표일 뿐 예측에 쓸 정보가 아님 |

특히 `Name`의 제외는 필수. 파일명은 행마다 값이 전부 다르므로 모델이 훈련 데이터를 통째로 외우는 것이 가능하며, 이는 규칙을 학습한 결과가 아님. 또한 악성 검체와 정상 검체를 서로 다른 경로에서 수집한 경우 명명 규칙 자체가 정답과 연관되어 **데이터 누수**로 이어질 가능성 존재.

**`to_numpy()`의 역할**

`DataFrame`을 NumPy 배열로 변환. scikit-learn은 입력을 내부에서 배열로 바꾸므로 이 호출이 필수는 아니지만, 자료형을 명시적으로 고정하는 효과 확보.

다만 변환 과정에서 **열 이름 정보가 사라짐**. 이후 변수 중요도를 확인할 때 열 번호와 원래 이름을 직접 대조해야 하므로, 해석이 중요한 분석에서는 `DataFrame` 상태 유지가 유리.

**자주 발생하는 오류**

끝의 괄호를 빠뜨려 `.to_numpy`로 쓰면 변환이 실행되지 않고 **함수 객체 자체**가 X에 저장. 파이썬에서 함수는 값처럼 다룰 수 있는 대상이라 이 자체로는 오류가 발생하지 않음. 문제는 다음 셀에서 이 객체를 숫자로 바꾸려 할 때 비로소 드러남.

```
TypeError: float() argument must be a string or a real number, not 'method'
```

오류가 표시된 셀과 실제 원인이 있는 셀이 다른 유형이므로, 오류 메시지 마지막 줄(`not 'method'`)을 단서로 앞 셀을 확인 필요.

In [3]:
from sklearn.preprocessing import StandardScaler

X_standardized = StandardScaler().fit_transform(X)

### 2. 특징 표준화

**하는 일**

각 열의 평균 $\mu$와 표준편차 $\sigma$를 구한 뒤 다음 식으로 값을 변환.

$$z = \frac{x - \mu}{\sigma}$$

변환 후 모든 열은 평균 0, 표준편차 1을 가짐. 값의 분포 모양 자체는 그대로이며 위치와 눈금 간격만 조정. 즉 데이터를 정규분포로 바꾸는 절차가 아님.

**필요한 이유**

PE 헤더 값은 열마다 크기 범위가 크게 다름. `SizeOfImage`는 백만 단위 바이트 값이고 `NumberOfSections`는 보통 한 자릿수. 이 상태로 학습하면 값이 큰 열이 계산 결과를 좌우.

| 표준화 필요 | 이유 |
|---|---|
| k-최근접이웃, SVM | 거리 계산이 값 큰 열에 좌우 |
| 경사하강법 기반 모델 | 수렴 속도 저하 |
| 주성분분석 | 분산이 큰 방향이 열의 단위에 좌우 |
| 릿지·라쏘 회귀 | 벌점 항이 계수 크기에 의존 |

반면 결정트리 계열(랜덤 포레스트, 그래디언트 부스팅)은 값의 크기가 아닌 대소 관계만 사용하므로 표준화 불필요.

**`fit`과 `transform`의 구분**

| 메서드 | 기능 |
|---|---|
| `fit(X)` | 열별 평균·표준편차를 계산해 `mean_`, `scale_`에 저장 |
| `transform(X)` | 저장된 값으로 변환 수행 |
| `fit_transform(X)` | 두 단계를 연속 수행 |

**보완 사항 — 전처리 단계의 데이터 누수**

현재 코드는 데이터를 나누기 전에 전체 X에 `fit_transform`을 적용. 이 경우 표준화에 쓰인 평균·표준편차에 테스트 데이터의 정보가 섞이며, 테스트 데이터가 더 이상 "본 적 없는 데이터"가 아니게 됨. 이를 **전처리 누수**라 하며 성능이 실제보다 높게 측정되는 원인.

올바른 순서는 분할을 먼저 하고, 훈련 데이터로만 `fit` 수행 후 나머지에는 `transform`만 적용.

```python
X_train, X_test = train_test_split(X, ...)
scaler = StandardScaler().fit(X_train)   # 평균·표준편차는 훈련 데이터에서만
X_train = scaler.transform(X_train)
X_test  = scaler.transform(X_test)       # fit 재실행 금지
```

교차검증을 함께 쓸 경우 `Pipeline`으로 전처리와 모델을 묶으면 각 조각마다 `fit`이 훈련 부분에만 적용되어 누수가 구조적으로 차단.

**참고 — 값이 모두 같은 열**

본 데이터에는 전체 행의 값이 동일해 표준편차가 0인 열이 8개 존재. `StandardScaler`는 이 경우 0으로 나누는 것을 피하려고 나눗셈 값을 1로 대체하지만, 변환 결과는 전부 0인 무의미한 열로 남음. `VarianceThreshold`로 사전 제거 권장.